# Generate unified .json with all extracted arguments 

- Clean duplicated arguments 
- Display summary statistics per SDG (totals/means, duplicates removed totals/means)
- Display summary statistics (same metrics over the whole set)

In [44]:
import os, re, json
from collections import OrderedDict
import pandas as pd
from typing import List, Optional, Tuple
import csv
import itertools

def process_sdg_runs(
    input_dir, output_dir, output_dir_stats,
    prefix,
    model_name,
    ods_range=(0, 17),
    unified_basename=None,
    interactive=True   # allow disabling manual review
):

    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(output_dir_stats, exist_ok=True)

    # 1. match files of interest
    model_part = re.escape(model_name) if model_name else r".+?"
    pattern = re.compile(rf"^{re.escape(prefix)}_ArgsSGD(\d+)_({model_part})\.json$")

    files = []
    for fn in os.listdir(input_dir):
        m = pattern.match(fn)
        if not m:
            continue
        ods_num = int(m.group(1))
        if ods_num < ods_range[0] or ods_num > ods_range[1]:
            continue
        files.append((ods_num, fn))

    if not files:
        raise FileNotFoundError(f"No files found in '{input_dir}' for prefix='{prefix}'"
                                + (f" and model='{model_name}'" if model_name else ""))

    # 2. Process per ODS (each .json file)
    files.sort(key=lambda x: (x[0], x[1]))

    unified_records = []
    per_sdg_stats = []

    for ods_num, fn in files:
        path = os.path.join(input_dir, fn)
        with open(path, "r", encoding="utf-8") as f:
            pages = json.load(f)

        seen_args = OrderedDict()
        not_valid_args = set()

        # Per-page stats
        page_counts_kept = []
        page_counts_removed = []
        page_counts_original = []
        page_counts_not_valid = []

        for item in pages:
            page = item.get("page")
            section = item.get("section")
            args: List[str] = item.get("arguments", []) or []

            def unique_in_order(seq):
                seen = set()
                out = []
                for s in seq:
                    if s not in seen:
                        out.append(s)
                        seen.add(s)
                return out

            args_original_unique = unique_in_order(args)
            page_original = len(args_original_unique)

            kept_args = []
            removed_here = 0
            not_valid_here = 0

            for a in args_original_unique:
                if a not in seen_args:
                    # manual revision
                    if interactive:
                        print(f"\n[ODS {ods_num} | page {page}] Argument candidate:\n  {a}")
                        resp = input("Keep this argument? (1=yes, 0=no): ").strip()
                        if resp == "0":
                            not_valid_args.add(a)
                            not_valid_here += 1
                            continue
                    kept_args.append(a)
                    seen_args[a] = True
                else:
                    removed_here += 1

            unified_records.append({
                "ods": ods_num,
                "page": page,
                "section": section,
                "arguments": kept_args,
                "source_file": fn,
            })

            page_counts_original.append(page_original)
            page_counts_kept.append(len(kept_args))
            page_counts_removed.append(removed_here)
            page_counts_not_valid.append(not_valid_here)

        # Per-SDG summary row
        pages_count = len(pages)
        total_kept = sum(page_counts_kept)
        total_removed = sum(page_counts_removed)
        total_original = sum(page_counts_original)
        total_not_valid = sum(page_counts_not_valid)

        per_sdg_stats.append({
            "prefix": prefix.replace("_", " ").strip(),
            "model": model_name,
            "ods": ods_num,
            "pages": pages_count,
            "total_args": total_original,
            "total_args_kept": total_kept,
            "total_dupes_removed": total_removed,
            "total_not_valid_args": total_not_valid,
        })

    per_sdg_df = pd.DataFrame(per_sdg_stats).sort_values(["ods"]).reset_index(drop=True)

    if unified_basename is None:
        unified_basename = f"{prefix}_AllArgs"

    unified_json_path = os.path.join(output_dir, f"{unified_basename}_{model_name}.json")
    per_sdg_csv_path = os.path.join(output_dir_stats, f"{unified_basename}_perSDG_summary.csv")

    # Save unified JSON
    with open(unified_json_path, "w", encoding="utf-8") as f:
        json.dump(unified_records, f, indent=2, ensure_ascii=False)

    # Append stats (instead of overwrite)
    if os.path.exists(per_sdg_csv_path):
        existing = pd.read_csv(per_sdg_csv_path)
        per_sdg_df = pd.concat([existing, per_sdg_df], ignore_index=True)
    per_sdg_df.to_csv(per_sdg_csv_path, index=False, encoding="utf-8")


    print(f"\nProcessed {len(per_sdg_df)} SDGs for prefix='{prefix}', model='{model_name}'")
    print(per_sdg_df.to_string(index=False))

    return {
        "unified_json": unified_json_path,
        "per_sdg_summary_csv": per_sdg_csv_path,
        "per_sdg_summary_df": per_sdg_df,
    }

import pandas as pd
import os

def save_overall_summary(res, output_dir_stats, model_name, unified_basename="AllArgs_overall_summary"):
    df = res["per_sdg_summary_df"]
    df = df[df["model"] == model_name]

    # Compute overall metrics
    overall_row = {
        "prefix": df["prefix"].iloc[0],
        "model": df["model"].iloc[0],
        "ods_max": df["ods"].max(),
        "pages_max": df["pages"].max(),
        "total_args": df["total_args"].sum(),
        "total_args_kept": df["total_args_kept"].sum(),
        "total_dupes_removed": df["total_dupes_removed"].sum(),
        "total_not_valid_args": df["total_not_valid_args"].sum(),
    }
    overall_df = pd.DataFrame([overall_row])
    overall_csv_path = os.path.join(output_dir_stats, f"{unified_basename}.csv")

    # Append if exists
    if os.path.exists(overall_csv_path):
        existing = pd.read_csv(overall_csv_path)
        overall_df = pd.concat([existing, overall_df], ignore_index=True)

    overall_df.to_csv(overall_csv_path, index=False, encoding="utf-8")

    print("\nUpdated overall summary:")

    return overall_df

def export_arguments_txt(json_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 1. Flat list of all arguments
    all_args = []
    for item in data:
        all_args.extend(item.get("arguments", []))

    flat_txt_path = os.path.join(output_dir, os.path.basename(json_path).replace(".json", "_arguments.txt"))
    with open(flat_txt_path, "w", encoding="utf-8") as f:
        for arg in all_args:
            f.write("'" + arg.strip() + "',\n")

    # 2. Dictionary by goal (ods)
    args_by_goal = {}
    for item in data:
        ods = item.get("ods")
        if ods not in args_by_goal:
            args_by_goal[ods] = []
        args_by_goal[ods].extend(item.get("arguments", []))

    dict_txt_path = os.path.join(output_dir, os.path.basename(json_path).replace(".json", "_by_goal.txt"))
    with open(dict_txt_path, "w", encoding="utf-8") as f:
        json.dump(args_by_goal, f, indent=2, ensure_ascii=False)

    print(f"Saved:\n - Flat arguments: {flat_txt_path}\n - Grouped by goal: {dict_txt_path}")
    return flat_txt_path, dict_txt_path


def create_relationship_csvs(path_by_goal, output_dir, model_name, prefix):
    # 1. Load arguments by goal
    with open(path_by_goal, "r", encoding="utf-8") as f:
        args_by_goal = json.load(f)

    os.makedirs(output_dir, exist_ok=True)

    # 2. Map args to unique IDs
    id_map = {}
    for goal, args in args_by_goal.items():
        for i, arg in enumerate(args):
            id_map[(goal, i)] = f"{goal}_{i}"

    intra_rows = []
    cross_rows = []

    # 3. Intra-goal relationships
    for goal, args in args_by_goal.items():
        for i, j in itertools.combinations(range(len(args)), 2):
            arg1, arg2 = args[i], args[j]
            id1, id2 = id_map[(goal, i)], id_map[(goal, j)]
            intra_rows.append([arg1, arg2, id1, id2, ""])

    # 4. Inter-goal relationships
    goals = list(args_by_goal.keys())
    for g1_idx in range(len(goals)):
        for g2_idx in range(g1_idx + 1, len(goals)):
            g1, g2 = goals[g1_idx], goals[g2_idx]
            for i, arg1 in enumerate(args_by_goal[g1]):
                for j, arg2 in enumerate(args_by_goal[g2]):
                    id1, id2 = id_map[(g1, i)], id_map[(g2, j)]
                    cross_rows.append([arg1, arg2, id1, id2, ""])

    # 5. Save CSVs
    intra_path = os.path.join(output_dir, "intra_goal" + prefix + model_name + ".csv")
    with open(intra_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["SDGarg1", "SDGarg2", "id_arg1", "id_arg2", "rel"])
        writer.writerows(intra_rows)

    cross_path = os.path.join(output_dir, "cross_goal" + prefix + model_name + ".csv")
    with open(cross_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["SDGarg1", "SDGarg2", "id_arg1", "id_arg2", "rel"])
        writer.writerows(cross_rows)

    print(f"Relationships intra-goal: {len(intra_rows)}")
    print(f"Relationships cross-goal: {len(cross_rows)}")

    return intra_path, cross_path



## GLOBAL SDG 2023 report

### Relevant sections to extract text from the Full report


| Section Title                                      | Pages  |
| -------------------------------------------------- | ------ |
| 1\_Executive Summary                               | 2      |
| 2\_How to Achieve the SDGs: The SDSN Framework     | 20     |
| 3\_The SDG Index and Dashboards                    | 6      |
| 4\_Government Efforts and Commitments for the SDGs | 12     |
| 5\_Lessons Learned and Next Steps                  | 13     |
| **Total**                                          | **53** |


### Qwen2.5-3b

In [ ]:
path_extracted_data = "Data\\Extracted Arguments No Keywords (all text)\\"
path_output = "Data\\Processed Arguments No Keywords\\"
path_output_stats = "Data\\Processed Arguments No Keywords\\Stats\\"
path_output_txt = "Data\\Processed Arguments No Keywords\\Args\\"
path_output_rels = "Data\\Relationships No Keywords\\"
prefix = 'GLOBAL_SGD2023_'
model_name="qwen2.5-3b"

res = process_sdg_runs(
    input_dir=path_extracted_data,
    output_dir=path_output,
    output_dir_stats=path_output_stats,
    prefix=prefix,
    model_name=model_name  
)

export_arguments_txt(
    json_path=res['unified_json'],
    output_dir=path_output_txt
)

print(f'Stats by SDG for {model_name} in document {prefix}:')
display(res['per_sdg_summary_df'])

print(f'Stats overall for {model_name} in document {prefix}:')
save_overall_summary(res, path_output_stats, model_name)

Stats by SDG for qwen2.5-3b in document GLOBAL_SGD2023_:


,prefix,model,ods,pages,total_args,total_args_kept,total_dupes_removed,total_not_valid_args
0,GLOBAL SGD2023,qwen2.5-3b,0,53,42,32,0,10
1,GLOBAL SGD2023,qwen2.5-3b,1,53,21,9,0,12
2,GLOBAL SGD2023,qwen2.5-3b,2,53,14,2,0,12
3,GLOBAL SGD2023,qwen2.5-3b,3,53,17,12,0,5
4,GLOBAL SGD2023,qwen2.5-3b,4,53,16,5,0,11
5,GLOBAL SGD2023,qwen2.5-3b,5,53,10,2,0,8
6,GLOBAL SGD2023,qwen2.5-3b,6,53,11,7,0,4
7,GLOBAL SGD2023,qwen2.5-3b,7,53,15,10,0,5
8,GLOBAL SGD2023,qwen2.5-3b,8,53,12,9,0,3
9,GLOBAL SGD2023,qwen2.5-3b,9,53,7,4,0,3


Stats overall for qwen2.5-3b in document GLOBAL_SGD2023_:

Updated overall summary:


,prefix,model,ods_max,pages_max,total_args,total_args_kept,total_dupes_removed,total_not_valid_args
0,GLOBAL SGD2023,qwen2.5-3b,17,53,251,161,0,90


In [41]:
path_output_rels = "Data\\Relationships No Keywords\\"
create_relationship_csvs(path_output_txt + prefix + '_AllArgs_' + model_name + '_by_goal.txt', 
                         path_output_rels,
                         model_name,
                         prefix)

Relationships intra-goal: 1058
Relationships cross-goal: 11822


('Data\\Relationships No Keywords\\intra_goalGLOBAL_SGD2023_qwen2.5-3b.csv',
 'Data\\Relationships No Keywords\\cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv')

### Gemma3 4B

In [ ]:
path_extracted_data = "Data\\Extracted Arguments No Keywords (all text)\\"
path_output = "Data\\Processed Arguments No Keywords\\"
path_output_stats = "Data\\Processed Arguments No Keywords\\Stats\\"
path_output_rels = "Data\\Relationships No Keywords\\"
prefix = 'GLOBAL_SGD2023_'
model_name="gemma3-4b"

res = process_sdg_runs(
    input_dir=path_extracted_data,
    output_dir=path_output,
    output_dir_stats=path_output_stats,
    prefix=prefix,
    model_name=model_name  
)

export_arguments_txt(
    json_path=res['unified_json'],
    output_dir=path_output_txt
)

create_relationship_csvs(path_output_txt + prefix + '_AllArgs_' + model_name + '_by_goal.txt', 
                         path_output_rels,
                         model_name,
                         prefix)

print(f'Stats by SDG for {model_name} in document {prefix}:')0

display(res['per_sdg_summary_df'])

print(f'Stats overall for {model_name} in document {prefix}:')
save_overall_summary(res, path_output_stats, model_name)


[ODS 0 | page 7] Argument candidate:
  At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.

[ODS 0 | page 7] Argument candidate:
  None of their objectives are beyond our reach.

[ODS 0 | page 7] Argument candidate:
  The SDGs are still achievable.

[ODS 0 | page 7] Argument candidate:
  It is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.

[ODS 0 | page 7] Argument candidate:
  To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.

[ODS 0 | page 7] Argument candidate:
  The Stimulus’ urgent objective is to address the chronic shortfall of international SDG financing confronting the LICs and LMICs, and to ramp up financing flows by at least US$500 billion by 2025.

[ODS 0 | page 7] Argument candidate:
  Greatly increase funding to national and subnational governments and private businesses, especi

,prefix,model,ods,pages,total_args,total_args_kept,total_dupes_removed,total_not_valid_args
0,GLOBAL_SGD2023_,qwen2.5-3b,0,53,42,32,0,10
1,GLOBAL_SGD2023_,qwen2.5-3b,1,53,21,9,0,12
2,GLOBAL_SGD2023_,qwen2.5-3b,2,53,14,2,0,12
3,GLOBAL_SGD2023_,qwen2.5-3b,3,53,17,12,0,5
4,GLOBAL_SGD2023_,qwen2.5-3b,4,53,16,5,0,11
5,GLOBAL_SGD2023_,qwen2.5-3b,5,53,10,2,0,8
6,GLOBAL_SGD2023_,qwen2.5-3b,6,53,11,7,0,4
7,GLOBAL_SGD2023_,qwen2.5-3b,7,53,15,10,0,5
8,GLOBAL_SGD2023_,qwen2.5-3b,8,53,12,9,0,3
9,GLOBAL_SGD2023_,qwen2.5-3b,9,53,7,4,0,3


Stats overall for gemma3-4b in document GLOBAL_SGD2023_:

Updated overall summary:


,prefix,model,ods_max,pages_max,total_args,total_args_kept,total_dupes_removed,total_not_valid_args
0,GLOBAL SGD2023,qwen2.5-3b,17,53,251,161,0,90
1,GLOBAL_SGD2023_,qwen2.5-3b,17,53,4344,867,2,3475


In [46]:
save_overall_summary(res, path_output_stats, model_name)


Updated overall summary:


,prefix,model,ods_max,pages_max,total_args,total_args_kept,total_dupes_removed,total_not_valid_args
0,GLOBAL SGD2023,qwen2.5-3b,17,53,251,161,0,90
1,GLOBAL SGD2023,gemma3-4b,17,53,4093,706,2,3385


### Gemma3 27B

In [ ]:
path_extracted_data = "Data\\Extracted Arguments No Keywords (all text)\\"
path_output = "Data\\Processed Arguments No Keywords\\"
path_output_stats = "Data\\Processed Arguments No Keywords\\Stats\\"
path_output_rels = "Data\\Relationships No Keywords\\"
prefix = 'GLOBAL_SGD2023_'
model_name="gemma3-27b"

res = process_sdg_runs(
    input_dir=path_extracted_data,
    output_dir=path_output,
    output_dir_stats=path_output_stats,
    prefix=prefix,
    model_name=model_name  
)

export_arguments_txt(
    json_path=res['unified_json'],
    output_dir=path_output_txt
)

create_relationship_csvs(path_output_txt + prefix + '_AllArgs_' + model_name + '_by_goal.txt', 
                         path_output_rels,
                         model_name,
                         prefix)

print(f'Stats by SDG for {model_name} in document {prefix}:')
display(res['per_sdg_summary_df'])

print(f'Stats overall for {model_name} in document {prefix}:')
save_overall_summary(res, path_output_stats)

### Llama 3.3 70B

In [ ]:
path_extracted_data = "Data\\Extracted Arguments No Keywords (all text)\\"
path_output = "Data\\Processed Arguments No Keywords\\"
path_output_stats = "Data\\Processed Arguments No Keywords\\Stats\\"
path_output_rels = "Data\\Relationships No Keywords\\"
prefix = 'GLOBAL_SGD2023_'
model_name="llama3.3-70b"

res = process_sdg_runs(
    input_dir=path_extracted_data,
    output_dir=path_output,
    output_dir_stats=path_output_stats,
    prefix=prefix,
    model_name=model_name  
)

export_arguments_txt(
    json_path=res['unified_json'],
    output_dir=path_output_txt
)

create_relationship_csvs(path_output_txt + prefix + '_AllArgs_' + model_name + '_by_goal.txt', 
                         path_output_rels,
                         model_name,
                         prefix)

print(f'Stats by SDG for {model_name} in document {prefix}:')
display(res['per_sdg_summary_df'])

print(f'Stats overall for {model_name} in document {prefix}:')
save_overall_summary(res, path_output_stats)

### Deepseek r1 70B

In [ ]:
path_extracted_data = "Data\\Extracted Arguments No Keywords (all text)\\"
path_output = "Data\\Processed Arguments No Keywords\\"
path_output_stats = "Data\\Processed Arguments No Keywords\\Stats\\"
path_output_rels = "Data\\Relationships No Keywords\\"
prefix = 'GLOBAL_SGD2023_'
model_name="deepseek-r1-70b"

res = process_sdg_runs(
    input_dir=path_extracted_data,
    output_dir=path_output,
    output_dir_stats=path_output_stats,
    prefix=prefix,
    model_name=model_name  
)

export_arguments_txt(
    json_path=res['unified_json'],
    output_dir=path_output_txt
)

create_relationship_csvs(path_output_txt + prefix + '_AllArgs_' + model_name + '_by_goal.txt', 
                         path_output_rels,
                         model_name,
                         prefix)

print(f'Stats by SDG for {model_name} in document {prefix}:')
display(res['per_sdg_summary_df'])

print(f'Stats overall for {model_name} in document {prefix}:')
save_overall_summary(res, path_output_stats)


## GLOBAL SDG 2025 report

| Section                                                | Pages  |
| ------------------------------------------------------ | ------ |
| 1\_Executive Summary                                   | 2      |
| 2\_Financing for Development                           | 9      |
| 3\_The SDG Index and Dashboards                        | 6      |
| 4\_Commitment to the SDGs and UN-Based Multilateralism | 10     |
| **Total**                                              | **27** |
